# Book Market Data Cleaning
Cleans `book_market_data_messy_Data.xlsx` to match the structure and logic of `book_market_intelligence.xlsx`.

In [1]:
import pandas as pd
import numpy as np

## 1. Load the Messy Data

In [6]:
df = pd.read_excel('book_market_data.xlsx')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

Shape: (1000, 8)

Columns: ['Title', 'Genre', 'Price', 'Price.1', 'Rating', 'Rating.1', 'Availability', 'Stock_Count']


,Title,Genre,Price,Price.1,Rating,Rating.1,Availability,Stock_Count
0,A Light in the Attic,Poetry,Â£51.77,£51.77,Three,3,In stock,22
1,Tipping the Velvet,Historical Fiction,Â£53.74,£53.74,One,1,In stock,20
2,Soumission,Fiction,Â£50.10,£50.10,One,1,In stock,20
3,Sharp Objects,Mystery,Â£47.82,£47.82,Four,4,In stock,20
4,Sapiens: A Brief History of Humankind,History,Â£54.23,£54.23,Five,5,In stock,20


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Title         1000 non-null   str  
 1   Genre         1000 non-null   str  
 2   Price         1000 non-null   str  
 3   Price.1       1000 non-null   str  
 4   Rating        1000 non-null   str  
 5   Rating.1      1000 non-null   int64
 6   Availability  1000 non-null   str  
 7   Stock_Count   1000 non-null   int64
dtypes: int64(2), str(6)
memory usage: 62.6 KB


## 2. Clean: Fix Encoding-Corrupted Price Column

`Price` has encoding artefacts (e.g. `Â£51.77`). `Price.1` is the correctly-encoded version. We keep `Price.1`, strip the `£` symbol, cast to float, and rename it `Price (£)`.

In [8]:
df['Price (£)'] = df['Price.1'].str.replace('£', '', regex=False).astype(float)
df.drop(columns=['Price', 'Price.1'], inplace=True)

print(df['Price (£)'].head())

0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: Price (£), dtype: float64


## 3. Clean: Convert Text Rating to Numeric

`Rating` contains word-based ratings (e.g. `Three`). `Rating.1` has the numeric equivalent. We keep the numeric column and rename it `Rating`.

In [9]:
df.drop(columns=['Rating'], inplace=True)   # drop the word-based text column
df.rename(columns={'Rating.1': 'Rating'}, inplace=True)  # promote numeric column

print(df['Rating'].value_counts().sort_index())

Rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64


## 4. Clean: Rename Stock Column

In [12]:
df.rename(columns={'Stock_Count': 'Stock Count'}, inplace=True)

## 5. Derive: Demand Analysis Column

Logic based on the target data:

| Rating | Stock Count | Label |
|--------|-------------|-------|
| 4–5    | 0–10        | Demand Gap |
| 4–5    | 11–22       | Well Supplied |
| 1–3    | 11–22       | Review Pricing |
| 1–3    | 0–10        | Low Priority |

In [13]:
def demand_analysis(row):
    high_rating = row['Rating'] >= 4
    high_stock  = row['Stock Count'] >= 11

    if high_rating and not high_stock:
        return 'Demand Gap'
    elif high_rating and high_stock:
        return 'Well Supplied'
    elif not high_rating and high_stock:
        return 'Review Pricing'
    else:
        return 'Low Priority'

df['Demand Analysis'] = df.apply(demand_analysis, axis=1)
df['Demand Analysis'].value_counts()

Demand Analysis
Low Priority      365
Review Pricing    260
Demand Gap        217
Well Supplied     158
Name: count, dtype: int64